# Validation with Pydantic

Last Tuesday, the value 20-45-67 appeared in the transaction amount
column. That's a sort code, not an amount. Nobody noticed for three
days. Three days of corrupted financial data, flowing into reports,
dashboards, and regulatory filings.

The old pipeline loaded it without complaint. Python is dynamically
typed -- it will happily put a string where a float should be. Pandas
will silently coerce the column to `object` type and carry on.

We need something that says "no" when the data is wrong. That
something is Pydantic.

## Setup

In [ ]:
%pip install -q pydantic

In [ ]:
import json
from datetime import datetime
from pydantic import BaseModel, Field, field_validator, ValidationError

## The problem

Let's load the incoming payments file and see the state of things.

In [ ]:
with open("../data/incoming_payments.json") as f:
    raw_payments = json.load(f)

print(f"{len(raw_payments)} payment records loaded")
raw_payments[0]

That first record looks fine. But let's hunt for trouble. What types
are lurking in the `amount` field?

In [ ]:
for i, payment in enumerate(raw_payments):
    if not isinstance(payment.get("amount"), (int, float)):
        print(f"Record {i} (id={payment['id']}): amount = {payment['amount']!r}")

Sort codes in the amount field. In a dynamically typed language with no
validation, these get loaded as strings and silently corrupt
downstream calculations. A `SUM()` in SQL might skip them. A pandas
mean will crash -- or worse, it won't, because pandas coerced the
whole column to strings and now the mean returns NaN with no
explanation.

The FCA would not be amused.

## Your first Pydantic model

A Pydantic model defines what a valid record looks like. You declare
the fields and their types, and Pydantic validates every record
against that schema.

If the data doesn't match, Pydantic raises a clear, detailed error.
No silent corruption. No guessing.

In [ ]:
class Payment(BaseModel):
    id: str
    amount: float
    currency: str
    sender_sort_code: str
    sender_account: str
    timestamp: str

That's it. Six fields, six types. Let's try validating a good record.

In [ ]:
good_record = {
    "id": "PAY0001",
    "amount": 150.00,
    "currency": "GBP",
    "sender_sort_code": "20-45-67",
    "sender_account": "12345678",
    "timestamp": "2024-11-15T10:30:00Z",
}

payment = Payment(**good_record)
print(payment)
print(f"Amount: {payment.amount}, type: {type(payment.amount)}")

Now let's try the sort-code-in-amount record.

In [ ]:
bad_record = {
    "id": "PAY0004",
    "amount": "20-45-67",  # This is a sort code, not an amount
    "currency": "GBP",
    "sender_sort_code": "30-12-89",
    "sender_account": "87654321",
    "timestamp": "2024-11-16T14:20:00Z",
}

try:
    Payment(**bad_record)
except ValidationError as e:
    print(e)

Pydantic rejects it instantly. The error message tells you exactly
what's wrong: the `amount` field received a string that cannot be
parsed as a float.

The old pipeline? It loaded this silently. Pydantic stops it at the
gate.

## Custom validators: business rules

Type checking is necessary but not sufficient. A value of `-500.00` is
a perfectly valid float, but it's not a valid payment amount. A
transaction dated next year is a valid timestamp, but it shouldn't be
in yesterday's batch.

Pydantic lets you add custom validators with `field_validator`. These
run after the type check passes, and they can enforce any business rule
you need.

In [ ]:
ALLOWED_CURRENCIES = {"GBP", "EUR", "USD"}


class ValidatedPayment(BaseModel):
    id: str
    amount: float
    currency: str
    sender_sort_code: str
    sender_account: str
    timestamp: str

    @field_validator("amount")
    @classmethod
    def amount_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError(f"Amount must be positive, got {v}")
        return v

    @field_validator("currency")
    @classmethod
    def currency_must_be_allowed(cls, v):
        if v not in ALLOWED_CURRENCIES:
            raise ValueError(f"Currency must be one of {ALLOWED_CURRENCIES}, got {v!r}")
        return v

    @field_validator("timestamp")
    @classmethod
    def timestamp_must_not_be_future(cls, v):
        dt = datetime.fromisoformat(v.replace("Z", "+00:00"))
        if dt > datetime.now(dt.tzinfo):
            raise ValueError(f"Timestamp is in the future: {v}")
        return v

Let's test each validation rule.

In [ ]:
# Negative amount
try:
    ValidatedPayment(
        id="PAY0012",
        amount=-33.25,
        currency="GBP",
        sender_sort_code="20-45-67",
        sender_account="12345678",
        timestamp="2024-11-20T09:00:00Z",
    )
except ValidationError as e:
    print("Negative amount caught:")
    print(e)

In [ ]:
# Invalid currency
try:
    ValidatedPayment(
        id="PAY9999",
        amount=100.00,
        currency="BTC",
        sender_sort_code="20-45-67",
        sender_account="12345678",
        timestamp="2024-11-20T09:00:00Z",
    )
except ValidationError as e:
    print("Invalid currency caught:")
    print(e)

In [ ]:
# Future timestamp
try:
    ValidatedPayment(
        id="PAY9998",
        amount=100.00,
        currency="GBP",
        sender_sort_code="20-45-67",
        sender_account="12345678",
        timestamp="2027-06-15T10:00:00Z",
    )
except ValidationError as e:
    print("Future timestamp caught:")
    print(e)

Three different problems, three clear error messages. Each validator
encodes a specific business rule, and the rules are declared right
there in the model definition. Anyone reading the code can see exactly
what constitutes a valid payment.

## Batch validation with error collection

In production, you don't stop at the first bad record. You validate
the entire batch and collect all the errors so you can report on them
at once.

The pattern is simple: loop through the records, try to validate each
one, and collect the successes and failures separately.

In [ ]:
valid_payments = []
errors = []

for i, record in enumerate(raw_payments):
    try:
        # Some records might be missing fields that the model requires,
        # so we only pass the fields we need (excluding nested 'merchant')
        flat_record = {
            "id": record.get("id"),
            "amount": record.get("amount"),
            "currency": record.get("currency"),
            "sender_sort_code": record.get("sender_sort_code"),
            "sender_account": record.get("sender_account"),
            "timestamp": record.get("timestamp"),
        }
        payment = ValidatedPayment(**flat_record)
        valid_payments.append(payment)
    except ValidationError as e:
        errors.append({"index": i, "id": record.get("id"), "errors": e.errors()})

print(f"Valid: {len(valid_payments)}")
print(f"Invalid: {len(errors)}")
print(f"Error rate: {len(errors) / len(raw_payments) * 100:.1f}%")

In [ ]:
# Show the first few errors
for err in errors[:5]:
    print(f"\nRecord {err['index']} (id={err['id']}):")
    for detail in err["errors"]:
        print(f"  Field: {detail['loc']}, Error: {detail['msg']}")

This gives you a complete picture. You know exactly how many records
failed, which records they were, and what was wrong with each one.
That information goes into a data quality report, and the bad records
get routed to a dead-letter queue for manual review.

The good records -- and only the good records -- proceed into the
system.

## Nested models

Real-world data is rarely flat. Our payment records include merchant
information as a nested object. Pydantic handles this naturally --
you define a model for the nested part and reference it in the
parent model.

In [ ]:
class Merchant(BaseModel):
    name: str
    category: str

    @field_validator("name")
    @classmethod
    def name_must_not_be_empty(cls, v):
        if not v or not v.strip():
            raise ValueError("Merchant name cannot be empty")
        return v


class FullPayment(BaseModel):
    id: str
    amount: float
    currency: str
    sender_sort_code: str
    sender_account: str
    timestamp: str
    merchant: Merchant

    @field_validator("amount")
    @classmethod
    def amount_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError(f"Amount must be positive, got {v}")
        return v

    @field_validator("currency")
    @classmethod
    def currency_must_be_allowed(cls, v):
        if v not in ALLOWED_CURRENCIES:
            raise ValueError(f"Currency must be one of {ALLOWED_CURRENCIES}, got {v!r}")
        return v

In [ ]:
# Good record with merchant info
good = raw_payments[0]
payment = FullPayment(**good)
print(f"Payment {payment.id}: {payment.amount} {payment.currency}")
print(f"Merchant: {payment.merchant.name} ({payment.merchant.category})")

In [ ]:
# Record with null merchant fields
bad_merchant_record = {
    "id": "PAY0056",
    "amount": 42.50,
    "currency": "GBP",
    "sender_sort_code": "20-45-67",
    "sender_account": "12345678",
    "timestamp": "2024-11-20T09:00:00Z",
    "merchant": {"name": None, "category": None},
}

try:
    FullPayment(**bad_merchant_record)
except ValidationError as e:
    print("Bad merchant caught:")
    print(e)

Pydantic validates the nested model too. If the merchant name is null
or empty, the whole record fails validation. The error message tells
you exactly where the problem is: `merchant -> name`.

## Why Pydantic and not just pandas dtypes?

You might be wondering: can't we just set pandas column types and let
pandas enforce them? In theory, yes. In practice, pandas is too
forgiving.

- pandas will silently coerce types (a string column becomes `object`)
- pandas has no concept of business rules (amount > 0, date not future)
- pandas errors are often cryptic and appear far downstream
- Pydantic errors are immediate, specific, and actionable

The right approach is to validate with Pydantic first, then load the
clean data into pandas for analysis. Validate at the boundary, trust
the data inside.

## Exercises

### Exercise 1

Add a `field_validator` to the `ValidatedPayment` model that checks
the `sender_sort_code` format. A valid UK sort code is exactly 8
characters in the format `XX-XX-XX` where X is a digit.

**Hint:** You can use a regular expression, or just check the length
and the positions of the hyphens.

### Exercise 2

Add a `field_validator` for `sender_account` that checks it is
exactly 8 digits. Test it with a valid account number and an
invalid one (e.g. "1234" or "ABCDEFGH").

### Exercise 3

Run the full batch validation using `FullPayment` (with nested
merchant validation) against all 200 records in `raw_payments`.
Collect the errors and print a summary showing:

1. How many records passed
2. How many records failed
3. The most common error type

### Exercise 4

Create a new Pydantic model called `StrictPayment` that adds a
`Field` constraint on the `amount`: it must be between 0.01 and
1,000,000 (inclusive). Use `Field(gt=0, le=1_000_000)` instead of
a custom validator.

Test it with amounts of 0, 50, and 2,000,000.

## Summary

- **Pydantic models** define what valid data looks like, using Python
  type hints
- **Type validation** catches the obvious problems: strings where
  numbers should be, missing fields, wrong types
- **`field_validator`** lets you encode business rules: positive
  amounts, allowed currencies, no future dates
- **Batch validation** processes an entire dataset and collects all
  errors -- you never stop at the first failure
- **Nested models** validate complex, hierarchical data structures
- Validate at the boundary, trust the data inside. Pydantic is
  the gatekeeper; pandas is the analyst.

In the next notebook, we'll deal with a different kind of data quality
problem: what happens when the schema itself changes.